# Notebook de Modelagem e Teste dos Dados

Neste notebook será realizada a etapa de avaliação dos dados já preparados no notebook de preparação (`data_prep.ipynb`).

Aqui vamos testar o desempenho dos nossos dados com **Random Forest** como modelo inicial de referência, por ser uma abordagem robusta e eficiente para classificação. A ideia é verificar se, com as features já tratadas, normalizadas e separadas em treino, validação e teste, conseguimos obter uma boa acurácia e um comportamento consistente na predição das classes de Libras.

Caso a acurácia do Random Forest não seja satisfatória, ou se houver espaço para melhoria, partiremos para o **XGBoost**, que costuma oferecer maior poder de ajuste e melhor desempenho em problemas de classificação mais complexos.

O foco deste notebook é comparar o desempenho dos modelos, validar os resultados com métricas adequadas e escolher a estratégia mais eficiente para a etapa final da pipeline.

## Fluxo passo a passo

1. **Carregar os dados já preparados**
   - Utilizar os arquivos gerados no `data_prep.ipynb`, como `train.csv`, `validation.csv` e `test.csv`.
   - Garantir que as features estejam padronizadas e prontas para treinamento.

2. **Entender a base final de entrada**
   - Conferir o formato dos dados.
   - Separar corretamente variáveis de entrada (`X`) e rótulo alvo (`y`).

3. **Treinar um modelo base com Random Forest**
   - Usar o conjunto de treino para ajustar o modelo.
   - Avaliar o resultado inicial na validação.

4. **Medir a performance com métricas de classificação**
   - Verificar acurácia, matriz de confusão e outras métricas relevantes.
   - Analisar se o modelo está generalizando bem.

5. **Decidir o próximo modelo com base no resultado**
   - Se a acurácia do Random Forest for boa, seguir com ele como solução principal.
   - Se o resultado ficar abaixo do esperado, testar o **XGBoost** como alternativa mais forte.

6. **Comparar Random Forest e XGBoost**
   - Avaliar qual modelo apresenta melhor desempenho geral.
   - Considerar não apenas acurácia, mas também equilíbrio entre classes e robustez.

7. **Validar o modelo escolhido no conjunto de teste**
   - Fazer a avaliação final apenas após a escolha do melhor modelo.
   - Registrar os resultados para uso na pipeline.

8. **Salvar conclusões e próximos passos**
   - Documentar o modelo selecionado.
   - Indicar possíveis melhorias futuras na preparação dos dados ou no ajuste de hiperparâmetros.

### Imports

In [14]:
#Visualização 
import matplotlib.pyplot as plt
import seaborn as sns


#Organização dos dados
import pandas as pd 
from pathlib import Path

#Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report


#Caminhos dos dados
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREPARED_DATA_DIR = PROJECT_ROOT / "data" / "data_output" / "prepared"

train_path = PREPARED_DATA_DIR / "train.csv"
test_path = PREPARED_DATA_DIR / "test.csv"
validate_path = PREPARED_DATA_DIR / "validation.csv"

### 1. Carregar Dados de Treino Já preparados

In [15]:
train_df = pd.read_csv(train_path)

train_df

,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z19,x20,y20,z20,source_file,dist_polegar_pulso,dist_indicador_pulso,dist_medio_pulso,dist_anelar_pulso,dist_minimo_pulso
0,F,0.0,0.0,0.0,-0.677623,-0.301028,0.317427,-0.567458,-0.373401,0.533034,...,0.194498,0.894385,-0.774290,-0.147177,datasetFSY (1).csv,-0.771304,-1.331987,0.806129,1.349503,0.665345
1,A,0.0,0.0,0.0,-1.250467,0.441103,-1.845972,-1.426928,-0.209762,-1.761458,...,-0.019638,0.182223,-0.212309,0.522337,datasetNBQ (1).csv,1.921355,-0.682275,-0.540615,-0.232819,-0.342223
2,O,0.0,0.0,0.0,-1.255528,-0.228857,0.756359,-0.964644,-0.273885,0.576929,...,0.036052,-1.894595,-0.374463,-0.116404,datasetLOV.csv,0.509741,0.262062,0.533861,1.076053,0.739861
3,L,0.0,0.0,0.0,-0.482106,0.286116,-0.041535,-0.654397,0.135690,-0.155060,...,0.255589,0.056500,0.359449,0.567356,datasetLOV.csv,-0.163514,0.966384,-1.588801,-1.099219,-1.144979
4,O,0.0,0.0,0.0,-1.282456,0.520329,1.102954,-1.334052,0.307999,1.042759,...,-0.345598,-1.818831,-0.392382,-0.512900,datasetLOV.csv,0.892670,0.371763,0.552107,0.998888,0.716125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19097,L,0.0,0.0,0.0,-0.516028,0.405059,-0.229214,-0.668147,0.274592,-0.258754,...,0.253225,0.051570,0.230817,0.477876,datasetLOV.csv,-0.280443,0.937185,-1.653777,-1.250012,-0.966328
19098,V,0.0,0.0,0.0,-0.200469,-0.181724,-0.434875,-0.072844,-0.148756,-0.438236,...,-0.257018,0.211421,0.433706,-0.268298,datasetLOV.csv,-1.131018,0.848339,0.847649,-0.781211,-1.118361
19099,S,0.0,0.0,0.0,1.759438,0.283400,-1.938952,2.007819,-0.149731,-1.844567,...,0.921189,-0.578695,-0.386973,1.579721,datasetFSY (1).csv,-0.458894,-0.336801,-0.382954,0.028362,-0.066640
19100,E,0.0,0.0,0.0,-0.565190,-0.021407,-0.085322,-0.660505,-0.212885,-0.090350,...,0.929974,0.136165,-0.581788,0.978107,datasetAMEP (1).csv,-0.863907,-0.714052,-0.323407,0.224278,0.151446


In [16]:
def preparar_features_e_alvo(df, label_column="label", indices_selecionados=None):
    if indices_selecionados is None:
        indices_selecionados = [0, 4, 8, 12, 16, 20]#colunas do punho e ponta dos dedos

    colunas_landmarks = [
        f"{dim}{idx}"
        for idx in indices_selecionados
        for dim in ("x", "y", "z")
    ]

    colunas_dist = [col for col in df.columns if "dist" in col.lower()]

    colunas_selecionadas = []
    for col in colunas_landmarks + colunas_dist:
        if col in df.columns and col not in colunas_selecionadas:
            colunas_selecionadas.append(col)

    x = df[colunas_selecionadas].copy()
    y = df[label_column].copy()

    return x, y, colunas_selecionadas

In [17]:
x_train, y_train, colunas_selecionadas = preparar_features_e_alvo(train_df)

print("Colunas selecionadas:", colunas_selecionadas)
print("Classe alvo (y):", y_train.name)

x_train.head()

Colunas selecionadas: ['x0', 'y0', 'z0', 'x4', 'y4', 'z4', 'x8', 'y8', 'z8', 'x12', 'y12', 'z12', 'x16', 'y16', 'z16', 'x20', 'y20', 'z20', 'dist_polegar_pulso', 'dist_indicador_pulso', 'dist_medio_pulso', 'dist_anelar_pulso', 'dist_minimo_pulso']
Classe alvo (y): label


,x0,y0,z0,x4,y4,z4,x8,y8,z8,x12,...,y16,z16,x20,y20,z20,dist_polegar_pulso,dist_indicador_pulso,dist_medio_pulso,dist_anelar_pulso,dist_minimo_pulso
0,0.0,0.0,0.0,0.039815,-0.286124,0.687273,-0.009718,-0.067236,0.220854,0.172574,...,-1.252646,-0.607613,0.894385,-0.774290,-0.147177,-0.771304,-1.331987,0.806129,1.349503,0.665345
1,0.0,0.0,0.0,-1.829089,-0.634176,-1.415037,-0.623131,-0.052253,-1.161749,-0.466531,...,-0.316044,0.369923,0.182223,-0.212309,0.522337,1.921355,-0.682275,-0.540615,-0.232819,-0.342223
2,0.0,0.0,0.0,-1.138327,-0.350373,0.256993,-1.140975,-0.357052,0.401211,-1.526726,...,-0.563132,0.069955,-1.894595,-0.374463,-0.116404,0.509741,0.262062,0.533861,1.076053,0.739861
3,0.0,0.0,0.0,-1.055721,0.324780,-0.417877,-0.452476,-0.898426,0.217279,-0.193421,...,0.216253,0.646300,0.056500,0.359449,0.567356,-0.163514,0.966384,-1.588801,-1.099219,-1.144979
4,0.0,0.0,0.0,-1.540069,-0.159071,0.492687,-1.658010,-0.056385,1.169950,-1.992050,...,-0.383277,-0.129321,-1.818831,-0.392382,-0.512900,0.892670,0.371763,0.552107,0.998888,0.716125


### Execução no terminal

```bash
# 1. Definimos o "espaço de busca" (as opções que queremos testar)
param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],       # Número de árvores
    'max_depth': [None, 5, 10, 20, 30],             # Profundidade máxima
    'min_samples_split': [2, 5, 10],                # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4],                  # Mínimo de amostras em cada folha
    'max_features': ['sqrt', 'log2', None]         # Quantidade de colunas testadas por árvore
}

# 2. Instanciamos o modelo base
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. Configuramos a busca aleatória
# n_iter=20 significa que ele vai testar 20 combinações aleatórias das opções acima
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,               # Validação cruzada em 5 partes (folds)
    random_state=42,
    n_jobs=-1,          # Usa todo o processador
    scoring='accuracy'  # Métrica de escolha (pode ser 'f1', 'roc_auc', etc.)
)

# 4. Rodamos a busca nos dados de treino
random_search.fit(x_train, y_train)

# --- RESULTADOS ---
print("🏆 Melhores Parâmetros Encontrados:")
print(random_search.best_params_)

saida: 

{'n_estimators': 100, 
 'min_samples_split': 2, 
 'min_samples_leaf': 1, 
 'max_features': 'sqrt', 
 'max_depth': 30}
```

In [18]:
#Definição do Modelo e dos parâmetros
model = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    max_depth= 30,
    
)

In [19]:
#Treinamento do Modelo
model.fit(x_train,y_train)
print("Treinamento Conluido")

Treinamento Conluido


In [20]:
#Carrega dataset de teste
test_df = pd.read_csv(test_path)

# Usa as mesmas colunas selecionadas no treino
x_test = test_df[colunas_selecionadas].copy()
y_test = test_df["label"].copy()



In [21]:
#Teste do modelo
y_pred = model.predict(x_test)

print("Resultados no conjunto de teste")
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall macro: {recall_score(y_test, y_pred, average='macro'):.4f}")



Resultados no conjunto de teste
Acurácia: 0.8487
F1-score macro: 0.8417
Recall macro: 0.8537


In [22]:
print("\nRelatório por classe:")
print(classification_report(y_test, y_pred))


Relatório por classe:
              precision    recall  f1-score   support

           A       0.95      0.99      0.97       272
           B       0.97      0.99      0.98       160
           C       0.99      1.00      1.00       149
           D       0.99      1.00      0.99       146
           E       0.97      0.91      0.94       166
           F       0.92      0.71      0.80       400
           G       1.00      1.00      1.00       146
           I       0.61      0.99      0.76       145
           L       0.97      1.00      0.98       178
           M       0.69      0.96      0.81       151
           N       0.98      0.61      0.75       158
           O       0.99      1.00      0.99       178
           P       0.97      0.97      0.97       171
           Q       0.99      1.00      0.99       147
           R       0.27      0.35      0.30       164
           S       1.00      0.99      0.99       398
           T       0.51      0.84      0.63       147
    